#**Practical No. 7**

**Aim:** To design and implement an ETL pipeline that extracts data from CSV and JSON files, cleans and transforms the data, identifies and removes invalid records, performs data validation, and loads the valid data into a relational database using incremental loading.

In [1]:
import pandas as pd
import sqlite3
import json

# 1. Create sample CSV data
a = pd.DataFrame({
    "id":[1,2,3,4],
    "name":["Amit","Priya","Rahul","Invalid"],
    "salary":[45000,40000,50000,-5000],
    "age":[24,25,28,150]
})

b = pd.DataFrame({
    "id":[5,6],
    "name":["Neha","Rohan"],
    "salary":[55000,42000],
    "age":[26,29]
})

a.to_csv("emp1.csv", index=False)
b.to_csv("emp2.csv", index=False)

# 2. Extract and combine CSV files
df = pd.concat([
    pd.read_csv("emp1.csv"),
    pd.read_csv("emp2.csv")
], ignore_index=True)

print("Combined Data:")
print(df)

# 3. Clean, validate and remove invalid records
df = df.drop_duplicates("id")
df = df[(df["salary"] > 0) & (df["age"].between(18,60))]

print("\nValid Data:")
print(df)

# 4. Connect to database
import os
if os.path.exists("employees.db"):
    os.remove("employees.db")
con = sqlite3.connect("employees.db")

# 5. Incremental loading
old = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con)
if "employees" in old["name"].values:
    ids = pd.read_sql("SELECT id FROM employees", con)["id"]
    df = df[~df["id"].isin(ids)]
df.to_sql("employees", con, if_exists="append", index=False)
# 6. Extract and transform JSON
data = [
    {"id":7,"name":"Karan","salary":60000},
    {"id":8,"name":"Anjali","salary":52000}
]

with open("emp.json","w") as f:
    json.dump(data,f)

json_df = pd.read_json("emp.json")
json_df["salary_lakh"] = json_df["salary"] / 100000

# 7. Load JSON into database
json_df.to_sql(
    "json_employees",
    con,
    if_exists="replace",
    index=False
)
# 8. Display result
print("\nEmployees Table:")
print(pd.read_sql("SELECT * FROM employees", con))

print("\nJSON Table:")
print(pd.read_sql("SELECT * FROM json_employees", con))
con.close()
print("\nETL Pipeline Completed Successfully!")

Combined Data:
   id     name  salary  age
0   1     Amit   45000   24
1   2    Priya   40000   25
2   3    Rahul   50000   28
3   4  Invalid   -5000  150
4   5     Neha   55000   26
5   6    Rohan   42000   29

Valid Data:
   id   name  salary  age
0   1   Amit   45000   24
1   2  Priya   40000   25
2   3  Rahul   50000   28
4   5   Neha   55000   26
5   6  Rohan   42000   29

Employees Table:
   id   name  salary  age
0   1   Amit   45000   24
1   2  Priya   40000   25
2   3  Rahul   50000   28
3   5   Neha   55000   26
4   6  Rohan   42000   29

JSON Table:
   id    name  salary  salary_lakh
0   7   Karan   60000         0.60
1   8  Anjali   52000         0.52

ETL Pipeline Completed Successfully!
